In [1]:
import pandas as pd
import json
import numpy as np 
import re
import seaborn as sns
%load_ext autoreload
%autoreload 2

# Compute rate of missingness in random sample between January 1, 2008 and Feb 28, 2018

## Missingness per patient

In [12]:
df = pd.read_csv('/Users/wayne/Desktop/Cluster/ClusterHome/Projects/2023/ClinicalNotes/MissingNotes/stats.csv')
missingCount_colnames = [x for x in df.columns if 'missing' in x]
totalCount_colnames = [x.replace('missing','total') for x in missingCount_colnames]
df['total_missing'] = df[missingCount_colnames].sum(axis=1)
df['total_notes'] = df[totalCount_colnames].sum(axis=1)
df['missing_rate'] =df['total_missing']/df['total_notes']
df['missing_rate'].fillna(0, inplace=True)

In [13]:
mask = df['missing_rate'] > 0
df.loc[ mask, ['total_missing','total_notes','missing_rate'] ]

,total_missing,total_notes,missing_rate
0,1,94,0.010638
3,1,14,0.071429


## Missingness per procedure name

In [19]:
proc_names = [x.replace('missing_', '') for x in missingCount_colnames]
dfMissingPerProcName = pd.DataFrame( { 'proc_name': proc_names, 'total_missing': df[missingCount_colnames].sum(axis = 0).values, 'total_count': df[totalCount_colnames].sum(axis=0).values } )
dfMissingPerProcName['missingPct'] = dfMissingPerProcName['total_missing']/dfMissingPerProcName['total_count']
dfMissingPerProcName['missingPct'].fillna(0, inplace=True)
dfMissingPerProcName.sort_values( by='missingPct', ascending=False )

,proc_name,total_missing,total_count,missingPct
7,telephone_advice,1,43,0.023256
0,clinic_note,1,281,0.003559
1,communication_note,0,84,0.000000
2,letter,0,130,0.000000
3,or_procedure_notes,0,56,0.000000
4,consultation_note,0,54,0.000000
5,radiation_therapy_note,0,28,0.000000
6,history_&_physical_note,0,2,0.000000
8,clinic_note_non-dictated,0,2,0.000000
9,discharge_summary,0,3,0.000000


In [20]:
# compute overall percentage of missing notes
print('Overall percentage of missing notes: ', dfMissingPerProcName['total_missing'].sum()/dfMissingPerProcName['total_count'].sum() )

Overall percentage of missing notes:  0.0029282576866764276


## Compute counts of notes per procedure name during this time period

In [17]:
dataDir = '/Users/wayne/Desktop/Cluster/H4Hhome/2BLAST/clinical_notes/HealthReportRecords/results_status_dates/processed/dataframes'
mergedNotes= pd.read_parquet(f'{dataDir}/merged_processed_clinicalNotes.parquet.gzip', engine='pyarrow', use_nullable_dtypes = True)
mergedNotes['visitDate'] = pd.to_datetime( mergedNotes['visitDate'], utc=True )

/var/folders/xl/pdrtfmy950768yqxtqn387s80000gp/T/ipykernel_4576/4212993288.py:2: FutureWarning: The argument 'use_nullable_dtypes' is deprecated and will be removed in a future version.Use dtype_backend='numpy_nullable' instead of use_nullable_dtype=True.
  mergedNotes= pd.read_parquet(f'{dataDir}/merged_processed_clinicalNotes.parquet.gzip', engine='pyarrow', use_nullable_dtypes = True)


In [23]:
mask = ( mergedNotes['visitDate'] >= '2008-1-01' ) & ( mergedNotes['visitDate'] < '2018-03-01' )
dfProcCount = mergedNotes.loc[ mask ].groupby(['Observations.ProcName']).size().reset_index(name='note_count')
dfProcCount['pct'] = dfProcCount['note_count']/dfProcCount['note_count'].sum()
dfProcCount.sort_values( by='pct', ascending=False, inplace=True )
dfProcCount

,Observations.ProcName,note_count,pct
0,Clinic Note,431258,0.479861
7,Letter,156519,0.174159
3,Consultation Note,72119,0.080247
9,OR Procedure/Notes,50321,0.055992
12,Radiation Therapy Note,49539,0.055122
2,Communication Note,45292,0.050396
14,Unscheduled Discharge Summary,40126,0.044648
13,Telephone Advice,32665,0.036346
1,Clinic Note (Non-dictated),12579,0.013997
5,Discharge Summary,5984,0.006658
